# Airline Customer Churn Prediction (Time-Series)

## Project Goal
Predict whether a customer will churn, where churn is defined as:

- **churn = 1** if the customer has **no flights for the next 6 consecutive months**
- **churn = 0** otherwise

This notebook is designed to be portfolio-ready and robust to extra columns in the dataset.

## Dataset
- Primary file: `fact_activity_cleaned.csv`
- Monthly customer activity table
- Expected columns include loyalty/customer identifiers, year/month, flights, distance, points, and redemption information

## Pipeline Overview
1. Data loading and schema profiling
2. Time-aware preparation and target creation
3. Feature engineering (recency, frequency, trend, engagement, drop indicators, aggregates)
4. EDA and churn behavior exploration
5. Model training (Logistic Regression, Random Forest, XGBoost)
6. Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix)
7. Feature importance and business insights

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")


def ensure_xgboost():
    try:
        from xgboost import XGBClassifier
        return XGBClassifier
    except Exception:
        print("xgboost not found. Installing now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
        from xgboost import XGBClassifier
        return XGBClassifier


print("Libraries loaded successfully.")

In [ ]:
# Locate dataset dynamically
candidate_paths = [
    Path.cwd() / "fact_activity_cleaned.csv",
    Path.cwd() / "07_ML" / "Airline Customer Churn Prediction" / "fact_activity_cleaned.csv",
    Path.cwd() / "Ailines project" / "07_ML" / "Airline Customer Churn Prediction" / "fact_activity_cleaned.csv",
    Path(__file__).resolve().parent / "fact_activity_cleaned.csv" if "__file__" in globals() else None
]
candidate_paths = [p for p in candidate_paths if p is not None]

csv_path = None
for p in candidate_paths:
    if p.exists():
        csv_path = p
        break

if csv_path is None:
    # fallback search
    search_root = Path.cwd()
    matches = list(search_root.rglob("fact_activity_cleaned.csv"))
    if matches:
        csv_path = matches[0]

if csv_path is None:
    raise FileNotFoundError("fact_activity_cleaned.csv not found. Place it in the project folder and rerun.")

print(f"Using dataset: {csv_path}")

df_raw = pd.read_csv(csv_path)
print(f"Shape: {df_raw.shape}")
display(df_raw.head(3))
print("\nColumns:")
print(df_raw.columns.tolist())

## 1. Data Preparation and Time Alignment

This section:
- Detects key columns dynamically
- Standardizes the monthly date index
- Builds a complete monthly panel per customer (fills missing months)
- Creates the churn target based on next 6 consecutive months of zero flights

In [ ]:
def detect_column(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in columns:
        c_low = c.lower()
        if any(cand.lower() in c_low for cand in candidates):
            return c
    return None


def preprocess_time_series(df):
    df = df.copy()

    id_col = detect_column(df.columns, ["loyalty_number", "customer_id", "customer_number", "member_id", "id"])
    year_col = detect_column(df.columns, ["year"])
    month_col = detect_column(df.columns, ["month"])
    flights_col = detect_column(df.columns, ["total_flights", "flights", "flight_count"])

    if id_col is None or year_col is None or month_col is None or flights_col is None:
        raise ValueError("Could not detect required columns: customer id, year, month, total flights")

    # Keep canonical references
    meta = {
        "id_col": id_col,
        "year_col": year_col,
        "month_col": month_col,
        "flights_col": flights_col,
        "distance_col": detect_column(df.columns, ["distance"]),
        "points_acc_col": detect_column(df.columns, ["points_accumulated", "points_earned", "points"]),
        "points_red_col": detect_column(df.columns, ["points_redeemed"]),
        "dollar_red_col": detect_column(df.columns, ["dollar_cost_points_redeemed", "dollar_redeemed", "cost_points_redeemed"])
    }

    # Date index at month granularity
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce")
    df[month_col] = pd.to_numeric(df[month_col], errors="coerce")
    df = df.dropna(subset=[year_col, month_col, id_col])

    df["date"] = pd.to_datetime(
        dict(year=df[year_col].astype(int), month=df[month_col].astype(int), day=1),
        errors="coerce"
    )
    df = df.dropna(subset=["date"]).sort_values([id_col, "date"]).reset_index(drop=True)

    # Build complete monthly panel per customer
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if flights_col not in num_cols:
        num_cols.append(flights_col)

    rebuilt = []
    for cust_id, g in df.groupby(id_col):
        g = g.sort_values("date").copy()
        full_months = pd.date_range(g["date"].min(), g["date"].max(), freq="MS")
        g = g.set_index("date").reindex(full_months)
        g.index.name = "date"
        g[id_col] = cust_id

        # numeric defaults
        for c in num_cols:
            if c not in g.columns:
                g[c] = 0

        # flights default 0 for missing months (critical for churn definition)
        g[flights_col] = pd.to_numeric(g[flights_col], errors="coerce").fillna(0)

        # other numeric activity fields also default 0 where missing
        for c in num_cols:
            if c == flights_col:
                continue
            g[c] = pd.to_numeric(g[c], errors="coerce").fillna(0)

        # non-numeric columns forward-fill within customer
        non_num = [c for c in g.columns if c not in num_cols + [id_col]]
        for c in non_num:
            g[c] = g[c].ffill().bfill()

        g = g.reset_index().rename(columns={"index": "date"})
        rebuilt.append(g)

    panel = pd.concat(rebuilt, ignore_index=True)
    panel["year"] = panel["date"].dt.year
    panel["month"] = panel["date"].dt.month

    return panel, meta


def create_churn_target(panel, meta, horizon=6):
    df = panel.copy()
    id_col = meta["id_col"]
    flights_col = meta["flights_col"]

    # Binary no-flight signal
    df["no_flight"] = (df[flights_col] <= 0).astype(int)

    # Check next 6 months no_flight all 1
    for k in range(1, horizon + 1):
        df[f"nf_lead_{k}"] = df.groupby(id_col)["no_flight"].shift(-k)

    lead_cols = [f"nf_lead_{k}" for k in range(1, horizon + 1)]
    df["churn"] = (df[lead_cols].sum(axis=1) == horizon).astype(int)

    # Rows without full future window are dropped from supervised modeling
    df["valid_target"] = df[lead_cols].notna().all(axis=1)

    return df


panel_df, meta = preprocess_time_series(df_raw)
model_base_df = create_churn_target(panel_df, meta, horizon=6)

print("Detected schema:")
for k, v in meta.items():
    print(f"- {k}: {v}")

print(f"\nPanel shape: {panel_df.shape}")
print(f"Model base shape: {model_base_df.shape}")
print("Churn distribution (valid rows only):")
display(model_base_df.loc[model_base_df["valid_target"], "churn"].value_counts(normalize=True).rename("ratio").to_frame())

## 2. Feature Engineering

Feature blocks implemented:
- Recency: months since last flight
- Frequency: rolling average flights (3m, 6m)
- Trends: rolling slope for flights and points
- Engagement: cumulative points and redemption ratio
- Activity drops: recent vs previous windows
- Customer-level aggregates
- Dynamic numeric lag/rolling features for additional columns

In [ ]:
def rolling_slope(values):
    y = np.asarray(values)
    x = np.arange(len(y))
    if len(y) < 2 or np.all(np.isnan(y)):
        return 0.0
    mask = ~np.isnan(y)
    y = y[mask]
    x = x[mask]
    if len(y) < 2:
        return 0.0
    return np.polyfit(x, y, 1)[0]


def engineer_features(df, meta):
    d = df.copy()
    id_col = meta["id_col"]
    flights_col = meta["flights_col"]
    points_acc_col = meta["points_acc_col"]
    points_red_col = meta["points_red_col"]

    d = d.sort_values([id_col, "date"]).reset_index(drop=True)

    # Recency: months since last positive flight
    d["had_flight"] = (d[flights_col] > 0).astype(int)
    d["flight_event_date"] = d["date"].where(d["had_flight"] == 1)
    d["last_flight_date"] = d.groupby(id_col)["flight_event_date"].ffill()
    d["months_since_last_flight"] = (
        (d["date"].dt.year - d["last_flight_date"].dt.year) * 12 +
        (d["date"].dt.month - d["last_flight_date"].dt.month)
    )
    d["months_since_last_flight"] = d["months_since_last_flight"].fillna(999)

    # Frequency
    d["avg_flights_last_3_months"] = d.groupby(id_col)[flights_col].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
    d["avg_flights_last_6_months"] = d.groupby(id_col)[flights_col].transform(
        lambda s: s.shift(1).rolling(6, min_periods=1).mean()
    )

    # Trend
    d["flight_trend"] = d.groupby(id_col)[flights_col].transform(
        lambda s: s.shift(1).rolling(6, min_periods=2).apply(rolling_slope, raw=False)
    )

    if points_acc_col is not None:
        d["points_trend"] = d.groupby(id_col)[points_acc_col].transform(
            lambda s: s.shift(1).rolling(6, min_periods=2).apply(rolling_slope, raw=False)
        )
    else:
        d["points_trend"] = 0.0

    # Engagement
    if points_acc_col is not None:
        d["total_points_accumulated"] = d.groupby(id_col)[points_acc_col].cumsum()
    else:
        d["total_points_accumulated"] = 0.0

    if points_acc_col is not None and points_red_col is not None:
        d["redemption_ratio"] = d[points_red_col] / (d[points_acc_col] + 1.0)
    elif points_red_col is not None:
        d["redemption_ratio"] = d[points_red_col] / 1.0
    else:
        d["redemption_ratio"] = 0.0

    # Activity drop indicators
    recent_3 = d.groupby(id_col)[flights_col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    prev_3 = d.groupby(id_col)[flights_col].transform(lambda s: s.shift(4).rolling(3, min_periods=1).mean())
    d["drop_in_flights"] = (recent_3 < prev_3).astype(int)
    d["drop_flights_delta"] = (recent_3 - prev_3).fillna(0)

    if points_acc_col is not None:
        p_recent = d.groupby(id_col)[points_acc_col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        p_prev = d.groupby(id_col)[points_acc_col].transform(lambda s: s.shift(4).rolling(3, min_periods=1).mean())
        d["drop_in_points"] = (p_recent < p_prev).astype(int)
        d["drop_points_delta"] = (p_recent - p_prev).fillna(0)
    else:
        d["drop_in_points"] = 0
        d["drop_points_delta"] = 0.0

    # Customer aggregates
    d["cust_avg_flights"] = d.groupby(id_col)[flights_col].transform("mean")
    d["cust_std_flights"] = d.groupby(id_col)[flights_col].transform("std").fillna(0)
    d["cust_max_flights"] = d.groupby(id_col)[flights_col].transform("max")

    if points_acc_col is not None:
        d["cust_avg_points_acc"] = d.groupby(id_col)[points_acc_col].transform("mean")

    # Dynamic additional features for all numeric columns (excluding target helpers)
    protected = {"churn", "valid_target", "year", "month", "no_flight"}
    numeric_cols = d.select_dtypes(include=[np.number]).columns.tolist()
    dynamic_base = [c for c in numeric_cols if c not in protected and not c.startswith("nf_lead_")]

    for col in dynamic_base:
        d[f"{col}_lag1"] = d.groupby(id_col)[col].shift(1)
        d[f"{col}_roll3"] = d.groupby(id_col)[col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

    # Cleanup
    drop_helper_cols = [c for c in d.columns if c.startswith("nf_lead_")] + ["flight_event_date", "last_flight_date", "had_flight"]
    d = d.drop(columns=[c for c in drop_helper_cols if c in d.columns])

    return d


feature_df = engineer_features(model_base_df, meta)
feature_df = feature_df[feature_df["valid_target"]].copy()

print(f"Feature dataframe shape: {feature_df.shape}")
display(feature_df.head(3))

## 3. Exploratory Data Analysis (EDA)

We inspect:
- Overall activity trends over time
- Core numeric distributions
- Churn vs non-churn behavior for major indicators

In [ ]:
id_col = meta["id_col"]
flights_col = meta["flights_col"]

monthly = feature_df.groupby("date", as_index=False).agg(
    total_flights=(flights_col, "sum"),
    active_customers=(id_col, "nunique"),
    churn_rate=("churn", "mean")
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.lineplot(data=monthly, x="date", y="total_flights", ax=axes[0])
axes[0].set_title("Total Flights Over Time")

sns.lineplot(data=monthly, x="date", y="active_customers", ax=axes[1])
axes[1].set_title("Active Customers Over Time")

sns.lineplot(data=monthly, x="date", y="churn_rate", ax=axes[2])
axes[2].set_title("Observed Churn Rate Over Time")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

core_num = [c for c in [flights_col, meta.get("distance_col"), meta.get("points_acc_col"), meta.get("points_red_col")] if c is not None and c in feature_df.columns]
if core_num:
    feature_df[core_num].hist(figsize=(12, 6), bins=30)
    plt.suptitle("Distributions of Core Activity Variables", y=1.02)
    plt.tight_layout()
    plt.show()

compare_cols = [c for c in ["months_since_last_flight", "avg_flights_last_6_months", "redemption_ratio", "drop_flights_delta"] if c in feature_df.columns]
if compare_cols:
    fig, axes = plt.subplots(1, len(compare_cols), figsize=(5 * len(compare_cols), 4))
    if len(compare_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, compare_cols):
        sns.boxplot(data=feature_df, x="churn", y=col, ax=ax)
        ax.set_title(f"{col} by Churn")
    plt.tight_layout()
    plt.show()

## 4. Modeling Dataset, Feature Selection, and Time-Based Split

- Build `X` and `y` dynamically from engineered features
- Use a **time-based split** (earliest 80% for train, latest 20% for test)
- Apply model-based feature selection using Random Forest importance

In [ ]:
exclude_cols = {
    "churn", "valid_target", "date", "year", "month",
    "no_flight", meta["id_col"]
}

candidate_features = [c for c in feature_df.columns if c not in exclude_cols]

X_all = feature_df[candidate_features].copy()
y_all = feature_df["churn"].astype(int).copy()

dates = feature_df["date"].copy()
cutoff_date = dates.quantile(0.8)

train_mask = dates <= cutoff_date
test_mask = dates > cutoff_date

X_train_raw, X_test_raw = X_all.loc[train_mask], X_all.loc[test_mask]
y_train, y_test = y_all.loc[train_mask], y_all.loc[test_mask]

dates_train, dates_test = dates.loc[train_mask], dates.loc[test_mask]

print(f"Cutoff date: {cutoff_date.date()}")
print(f"Train size: {X_train_raw.shape}, Positive churn ratio: {y_train.mean():.3f}")
print(f"Test size: {X_test_raw.shape}, Positive churn ratio: {y_test.mean():.3f}")

num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train_raw.columns if c not in num_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

# Model-based feature selection on transformed train data
X_train_trans = preprocessor.fit_transform(X_train_raw)
X_test_trans = preprocessor.transform(X_test_raw)

feature_names_num = num_cols
feature_names_cat = []
if len(cat_cols) > 0:
    ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    feature_names_cat = ohe.get_feature_names_out(cat_cols).tolist()

feature_names = feature_names_num + feature_names_cat

selector_model = RandomForestClassifier(
    n_estimators=250,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    n_jobs=-1
)
selector_model.fit(X_train_trans, y_train)

importances = pd.Series(selector_model.feature_importances_, index=feature_names).sort_values(ascending=False)
keep_n = min(60, len(importances))
selected_features = importances.head(keep_n).index.tolist()
selected_idx = [feature_names.index(f) for f in selected_features]

X_train_sel = X_train_trans[:, selected_idx]
X_test_sel = X_test_trans[:, selected_idx]

print(f"Selected top {len(selected_features)} features.")
display(importances.head(15).to_frame("importance"))

## 5. Train Models

Models trained:
- Logistic Regression
- Random Forest
- XGBoost

In [ ]:
XGBClassifier = ensure_xgboost()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    fitted_models[name] = model

print("All models trained.")

## 6. Evaluation

Metrics reported:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

In [ ]:
results = []
pred_store = {}

for name, model in fitted_models.items():
    y_pred = model.predict(X_test_sel)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_sel)[:, 1]
    else:
        y_proba = model.decision_function(X_test_sel)

    metrics_row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }
    results.append(metrics_row)
    pred_store[name] = (y_pred, y_proba)

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(results_df)

best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
best_pred, best_proba = pred_store[best_model_name]

print(f"Best model by ROC-AUC: {best_model_name}")
print("\nClassification Report (best model):")
print(classification_report(y_test, best_pred, digits=3))

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
for name, (_, proba) in pred_store.items():
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)
plt.title("ROC Curves")
plt.show()

## 7. Model Interpretation and Churn Drivers

This section extracts key feature importance from the best model to identify behavior patterns associated with churn risk.

In [ ]:
importance_df = None

if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": selected_features,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

elif hasattr(best_model, "coef_"):
    coef = np.abs(best_model.coef_).ravel()
    importance_df = pd.DataFrame({
        "feature": selected_features,
        "importance": coef
    }).sort_values("importance", ascending=False)

if importance_df is not None and not importance_df.empty:
    top_imp = importance_df.head(20)
    display(top_imp)

    plt.figure(figsize=(8, 6))
    sns.barplot(data=top_imp.sort_values("importance"), x="importance", y="feature")
    plt.title(f"Top Feature Drivers - {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("No direct feature importance available for this model.")

## 8. Business Insights and High-Risk Customer Identification

Actionable output:
- Main churn behavior patterns
- Earliest warning indicators
- Ranked high-risk customers for retention actions

In [ ]:
scored = feature_df.loc[test_mask, [meta['id_col'], 'date', 'churn']].copy()
scored["pred_churn"] = best_pred
scored["churn_probability"] = best_proba

high_risk = scored.sort_values("churn_probability", ascending=False).head(20)
display(high_risk)

# Auto-generated business insight bullets
insights = []
insights.append(f"Best model: {best_model_name} (ROC-AUC={results_df.iloc[0]['roc_auc']:.3f}, F1={results_df.iloc[0]['f1']:.3f})")

if importance_df is not None and not importance_df.empty:
    top_features = importance_df.head(5)["feature"].tolist()
    insights.append("Top churn drivers: " + ", ".join(top_features))

insights.append("Customers with high months_since_last_flight and declining flight trend are likely churn candidates.")
insights.append("Redemption and points trend features help identify weakening engagement before complete inactivity.")
insights.append("Retention playbook: trigger outreach when risk score rises and recent 3-month activity drops vs previous period.")

print("Business Insights:")
for i, line in enumerate(insights, 1):
    print(f"{i}. {line}")

# Save project outputs
output_dir = Path.cwd() / "outputs_churn"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "model_comparison.csv", index=False)
high_risk.to_csv(output_dir / "high_risk_customers_top20.csv", index=False)
if importance_df is not None:
    importance_df.to_csv(output_dir / "feature_importance.csv", index=False)

print(f"\nSaved outputs to: {output_dir}")